## Go through the 5.1 µ Jupiter images for the SPEX instrument for the 2024 observation cycles.
### The spex_interactive notebook was an explicit way of checking the images, but here we will run through them all and generate a list of images that satisfy the longitudinal overlap needed for the zonal wind correlation.

### Output: a .txt or .csv file that can be fed to the zonal wind code.

In [3]:
import numpy as np
import sys
from astropy.io import fits
import os
import glob
import datetime
import csv
import re


In [4]:
main_path = 'spex/'
pattern   = 'jc*[0-9][0-9].cmap.gz' # assumes all relevant fits files start with jc... (some files were named cj** but they are irrelevant)
pattern_2 = 'jc*[0-9][0-9].cmap.fits.gz' # the "other" pattern; only due to inconsistent naming, not special itself.

two_J_days = 9.9258 * 2 # Two full Jovian rotations assuming 9 h 55 m 33 s for the synodic rotation period
thr_J_days = 9.9258 * 3 # Three full Jovian rotations assuming 9 h 55 m 33 s for the synodic rotation period

print(two_J_days, two_J_days*3600)
print(thr_J_days, thr_J_days*3600)

19.8516 71465.76000000001
29.7774 107198.64


In [5]:
date_pair = [['2024feb5 ', '2024feb6  '], 
             ['2024jul15', '2024jul16'], 
             ['2024aug16', '2024aug17'],
             ['2024sep19', '2024sep20'],
             ['2024oct22', '2024oct23'],
             ['2024nov23', '2024nov24']]


def getTemporalDifference(file_list):

    # returns the total seconds elapsed between observations
    d_obj = []
    for i_f, f in enumerate(file_list):
        try:
            with fits.open(f) as hdul:
                hdr = hdul[0].header
                date_obs_str = hdr['DATE_OBS'].split("-")
                time_obs_str = hdr['TIME_OBS'].split(":")

                # Extract seconds and milliseconds (fractional part)
                seconds_full = float(time_obs_str[2])
                seconds = int(seconds_full)  # Integer part of seconds
                microseconds = int((seconds_full - seconds) * 1e6)  # Convert fractional seconds to microseconds
                # Parse datetime
                d_obj.append(datetime.datetime(
                    int(date_obs_str[0]), int(date_obs_str[1]), int(date_obs_str[2]),
                    int(time_obs_str[0]), int(time_obs_str[1]), seconds, microseconds
                ))
        except ValueError as e:
            print(f"Error parsing file {i_f}: {f}, Error: {e}")

    return (d_obj[-1] - d_obj[0]).total_seconds()


def DPpaths(date_pair, hour_thresh=0.75, see_main_files=False):
    '''
    Make the binary date pairs for each file so the time difference can be computed readily.
    If see_main_files=False, outputs a populated "validity_table" with paths to all files that satisfy the temporal criteria set by hour_thresh
    '''
    validity_table = []

    # Case-insensitive regex pattern for file filtering
    regex = re.compile(r'jc.*[0-9][0-9]\.cmap(\.fits)?\.gz$', re.IGNORECASE)
    
    for d in date_pair:
        d0_path  = main_path + d[0].strip()
        d1_path  = main_path + d[1].strip()
        
        # files_d0 = glob.glob(d0_path + '/' + pattern)
        # files_d1 = glob.glob(d1_path + '/' + pattern)
        # check if the files were named **.fits.gz instead of **.gz
        # if len(files_d0) == 0:
        #     files_d0 = glob.glob(d0_path + '/' + pattern_2)
        # if len(files_d1) == 0:
        #     files_d1 = glob.glob(d1_path + '/' + pattern_2)

        # Get all files and filter using regex (no need for pattern_1 or pattern_2 with this one)
        files_d0 = [f for f in glob.glob(d0_path + '/*') if regex.search(f)]
        files_d1 = [f for f in glob.glob(d1_path + '/*') if regex.search(f)]

        if see_main_files: # if you just want to see the files themselves
            print(d[0])
            for f0 in files_d0:
                with fits.open(f0) as hdul:
                    hdr = hdul[0].header
                    print(f0, hdr.get('DATE_OBS', 'N/A'), hdr.get('TIME_OBS', 'N/A'))
            print(d[1])
            for f1 in files_d1:
                with fits.open(f1) as hdul:
                    hdr = hdul[0].header
                    print(f1, hdr.get('DATE_OBS', 'N/A'), hdr.get('TIME_OBS', 'N/A'))
            print()
        else: # otherwise, print the image pairs along with their time differences in hours
            for i in files_d0:
                for j in files_d1:
                    t_diff_hrs = getTemporalDifference([i, j]) / 3600
                    # Too close in time - will not have much longitudinal overlap
                    if t_diff_hrs < 10:
                        print(i, j, t_diff_hrs, ' Date overlap detected - Not usable...')
                    # image pair is about 2 Jupiter rotations apart
                    elif abs(t_diff_hrs - two_J_days) <= hour_thresh:
                        print(i, j, t_diff_hrs, '\t VALID - 2 JUPITER ROTATIONS')
                        validity_table.append([i, j, float(t_diff_hrs)])
                    # image pair is about 3 Jupiter rotations apart
                    elif abs(t_diff_hrs - thr_J_days) <= hour_thresh:
                        print(i, j, t_diff_hrs, '\t VALID - 3 JUPITER ROTATIONS')
                        validity_table.append([i, j, float(t_diff_hrs)])
                    # image pair is separated by an inconvenient timeframe - cannot be used due to little overlap or too far apart in time.
                    else:
                        print(i, j, t_diff_hrs)
            print()
    if see_main_files: # if you just want to see the files themselves
        return None
    else:
        print(f"Number of pairs usable for zonal winds (within {hour_thresh*60} minutes): {len(validity_table)}")
        return validity_table

# def Remove_cal_crossers(VT_inhomo, path_to_VT_inhomo):
#     '''
#     cal_ is reserved for images that have been photometrically calibrated. Comparing 
#     Removes cal_ crossers but does not address the jc* vs. jcf* files. That problem is less common than the cal_ crossers.
#     '''
#     final_table = []
#     with open(path_to_VT_inhomo, "r") as f:
#         reader = csv.reader(f)
#         for row in reader:
#             row0, row1 = row[0].split('cal_'), row[1].split('cal_')
#             if len(row0) == len(row1):
#                 final_table.append([row[0], row[1], float(row[2])])
#             else:
#                 pass
#     # Now, the 'cal_' has been homogenized but there may still be repeaters as the non-calibrated image is also still included.
#     # Thus, cal_jcf12344* is treated as the same as jcf12344*
#     final_table_reshaped = np.reshape(final_table, (len(final_table), 3))
#     for i in range(final_table_reshaped.shape[0]):
#         for j in range(final_table_reshaped.shape[0]):
#             if i!=j and np.isclose(final_table_reshaped[i, -1], final_table_reshaped[j, -1], 0.0, 1e-10):
                
    # return final_table_reshaped

In [6]:
hdul = fits.open(main_path + date_pair[0][0].strip() + '/jc118152.fits.gz')
hdr = hdul[0].header

print(hdr['DATE_OBS'], hdr['TIME_OBS'])

2024-02-05 00:19:18.21533


In [7]:
# print(os.listdir(os.getcwd()+ '/' + main_path))
DPpaths(date_pair, 0.85, True)

2024feb5 
spex/2024feb5/jc118152.cmap.fits.gz 2024-02-05 00:19:18.21533
spex/2024feb5/jc587611.cmap.fits.gz 2024-02-05 03:53:37.46973
spex/2024feb5/jc311333.cmap.fits.gz 2024-02-05 01:28:17.11084
2024feb6  
spex/2024feb6/jc351385.cmap.fits.gz 2024-02-06 00:58:59.26978
spex/2024feb6/jc187221.cmap.fits.gz 2024-02-06 00:09:08.07947
spex/2024feb6/jc564599.cmap.fits.gz 2024-02-06 03:02:15.77246

2024jul15
spex/2024jul15/jc229256.cmap.fits.gz 2024-07-15 16:47:19.8125
spex/2024jul15/jc845850.cmap.fits.gz 2024-07-15 19:46:17.5625
spex/2024jul15/jc611710.cmap.fits.gz 2024-07-15 18:26:35.80469
spex/2024jul15/jc315388.cmap.fits.gz 2024-07-15 17:27:56.15234
2024jul16
spex/2024jul16/jc599608.cmap.fits.gz 2024-07-16 18:31:28.66406
spex/2024jul16/jc953984.cmap.fits.gz 2024-07-16 20:40:33.35938
spex/2024jul16/jc357362.cmap.fits.gz 2024-07-16 16:58:36.86328
spex/2024jul16/jc871882.cmap.fits.gz 2024-07-16 20:20:09.67969
spex/2024jul16/jc673702.cmap.fits.gz 2024-07-16 18:44:11.57812
spex/2024jul16/jc4114

In [8]:
VT = DPpaths(date_pair, 0.85, False)
VT_reshaped = np.reshape(VT, (len(VT), 3))
# print(getTemporalDifference(['spex/2024feb5/jc118152.gz', 'spex/2024feb5/jc311333.gz']) / 3600) # returns expected time difference

spex/2024feb5/jc118152.cmap.fits.gz spex/2024feb6/jc351385.cmap.fits.gz 24.66140401361111
spex/2024feb5/jc118152.cmap.fits.gz spex/2024feb6/jc187221.cmap.fits.gz 23.830517816666667
spex/2024feb5/jc118152.cmap.fits.gz spex/2024feb6/jc564599.cmap.fits.gz 26.715988091666667
spex/2024feb5/jc587611.cmap.fits.gz spex/2024feb6/jc351385.cmap.fits.gz 21.08938890277778
spex/2024feb5/jc587611.cmap.fits.gz spex/2024feb6/jc187221.cmap.fits.gz 20.258502705833333 	 VALID - 2 JUPITER ROTATIONS
spex/2024feb5/jc587611.cmap.fits.gz spex/2024feb6/jc564599.cmap.fits.gz 23.143972980833333
spex/2024feb5/jc311333.cmap.fits.gz spex/2024feb6/jc351385.cmap.fits.gz 23.511710816666664
spex/2024feb5/jc311333.cmap.fits.gz spex/2024feb6/jc187221.cmap.fits.gz 22.68082461972222
spex/2024feb5/jc311333.cmap.fits.gz spex/2024feb6/jc564599.cmap.fits.gz 25.566294894722223

spex/2024jul15/jc229256.cmap.fits.gz spex/2024jul16/jc599608.cmap.fits.gz 25.73579209972222
spex/2024jul15/jc229256.cmap.fits.gz spex/2024jul16/jc953984.

In [9]:
print(VT_reshaped)
a, b = 20.258502705833333, 20.2585027059
print(float(VT_reshaped[0, -1]), np.isclose(a, b, 0.0, 1e-13), abs(a-b))

[['spex/2024feb5/jc587611.cmap.fits.gz'
  'spex/2024feb6/jc187221.cmap.fits.gz' '20.258502705833333']
 ['spex/2024aug16/jc523554.cmap.fits.gz'
  'spex/2024aug17/jc042065.cmap.fits.gz' '20.52804470555556']
 ['spex/2024aug16/jc649676.cmap.fits.gz'
  'spex/2024aug17/jc148182.cmap.fits.gz' '20.54506293333333']
 ['spex/2024aug16/jc649676.cmap.fits.gz'
  'spex/2024aug17/jc042065.cmap.fits.gz' '20.091280380277777']
 ['spex/2024aug16/jc183201.cmap.fits.gz'
  'spex/2024aug17/jc148182.cmap.fits.gz' '19.373233508333332']
 ['spex/2024aug16/jc183201.cmap.fits.gz'
  'spex/2024aug17/jc329353.cmap.fits.gz' '20.205661894444443']
 ['spex/2024aug16/jc815843.cmap.fits.gz'
  'spex/2024aug17/jc148182.cmap.fits.gz' '19.372280816944443']
 ['spex/2024aug16/jc815843.cmap.fits.gz'
  'spex/2024aug17/jc329353.cmap.fits.gz' '20.204709203055558']
 ['spex/2024oct22/jc505516.cmap.fits.gz'
  'spex/2024oct23/jc057087.cmap.fits.gz' '20.521971571944448']
 ['spex/2024oct22/jc451486.cmap.fits.gz'
  'spex/2024oct23/jc057087.

In [10]:
# np.savetxt('spex/pairs_2024_45min.txt', VT_reshaped)

# Save to a pickle file

def saveFile(path_file, str_table, pkl_form=None):

    if pkl_form==None:
        # Save to a CSV file
        import csv
        with open(path_file+".csv", "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerows(str_table)
    elif pkl_form:
        import pickle
        # Save to a pickle file
        with open(path_file+".pkl", "wb") as f:
            pickle.dump(str_table, f)

# saveFile(main_path+'pairs_2024_51min_3rots', VT_reshaped, pkl_form=None)

# Load from a pickle file
# with open("spex/pairs_2024_45min.pkl", "rb") as f:
#     loaded_list = pickle.load(f)

# print(loaded_list)

# Load from a CSV file
with open(main_path+"pairs_2024_51min_cmap_3rots.csv", "r") as f:
    reader = csv.reader(f)
    for row in reader:
        selected_pair, t_diff_of_pair = [row[0], row[1]], float(row[2])

In [11]:
print(VT_reshaped[-1])
print(selected_pair)
print(t_diff_of_pair)

['spex/2024nov23/jc031037.cmap.fits.gz'
 'spex/2024nov24/jc763767.cmap.fits.gz' '29.585512152777778']
['spex/2024nov23/jc031037.cmap.fits.gz', 'spex/2024nov24/jc763767.cmap.fits.gz']
29.585512152777778


'''
All date_pair lists for 5µ IRTF data using spex:
'''

# # for 2024 - 5µ
date_pair_2024 = [['2024feb5 ', '2024feb6  '], 
             ['2024jul15', '2024jul16'], 
             ['2024aug16', '2024aug17'],
             ['2024sep19', '2024sep20'],
             ['2024oct22', '2024oct23'],
             ['2024nov23', '2024nov24']]

# # for 2023 - 5µ
date_pair_2023 = [['2023dec28', '2023dec29'], 
             ['2023jul29', '2023jul30'], 
             ['2023jun23', '2023jun24'],
             ['2023may16', '2023may17'],
             ['2023nov22', '2023nov23'],
             ['2023oct14', '2023oct15']]

# # 2022 - 5µ
date_pair_2022 = [['2022apr29', '2022apr30'], 
             ['2022apr8 ', '2022apr9 '], 
             ['2022aug17', '2022aug18'],
             ['2022jan14', '2022jan15'],
             ['2022jul25', '2022jul26'],
             ['2022jul8 ', '2022jul9 '],
             ['2022jun11', '2022jun12'],
             ['2022may21', '2022may22']]

# # for 2021 - 5µ
date_pair_2021 = [['2021dec17', '2021dec18'], 
             ['2021jul21', '2021jul22'], 
             ['2021jun24', '2021jun25'],
             ['2021jun7 ', '2021jun8 '],
             ['2021may26', '2021may27'],
             ['2021oct17', '2021oct18'],
             ['2021sep21', '2021sep22'],
             ['2021sep4 ', '2021sep5 ']]

# for 2020 - 5µ 
date_pair_2020 = [['2020dec23', '2020dec24'], # Tom Momary needs to enable read permissions for dec... TO DO!!!
             ['2020jul4 ', '2020jul5 '], 
             ['2020nov10', '2020nov11'],
             ['2020nov6 ', '2020nov7 '],
             ['2020nov7 ', '2020nov8 '],
             ['2020oct14', '2020oct15'],
             ['2020oct3 ', '2020oct4 '],
             ['2020sep17', '2020sep18'],
             ['2020sep21', '2020sep22']]

# Also for 2020 - 5µ, Dir: 2020aug15/5.10/jupiter/* not 2020aug15/images/5.10/jupiter/* TO DO still because 2020 is not yet added.
date_pair_2020_add = [['2020aug15', '2020aug16']] --> not been reduced!

# for 2019 - 5µ # Although only two nights, these dates have a ton of data! Inquire Glenn.
date_pair_2019 = [['2019jan16', '2019jan17']]

# Also for 2019 - 5µ, Dir: 2019aug27/5.10/jupiter/* not 2019aug27/images/5.10/jupiter/*.
date_pair_2019_add = [['2019aug27', '2019aug28'],
                      ['2019jan5 ', '2019jan6 '], 
                      ['2019jan6 ', '2019jan7 '],
                      ['2019jan7 ', '2019jan8 '],
                      ['2019jan8 ', '2019jan9 '],
                      ['2019nov4 ', '2019nov5 '],
                      ['2019sep10', '2019sep11'],
                      ['2019sep18', '2019sep19'],
                      ['2019sep7 ', '2019sep8 ']]



In [5]:

test_str  = 'jdscsfc/cal_jc2020.cmap.gz'
test_str1 = 'jdscsfc/jc2020.cmap.gz'
print(len(test_str.split('cal_')), len(test_str1.split('cal_')))

2 1


In [16]:
with open("rx2019_5.10_add.csv", "r") as f:
    reader = csv.reader(f)
    for row in reader:
        print(row)

["[re.compile(r'^jcf.*?[0-9]{2}\\.cmap(\\.fits)?\\.gz$'", ' re.IGNORECASE)', " re.compile(r'^jcf.*?[0-9]{2}\\.cmap(\\.fits)?\\.gz$'", ' re.IGNORECASE)]', '']
["[re.compile(r'^jcf.*?[0-9]{2}\\.cmap(\\.fits)?\\.gz$'", ' re.IGNORECASE)', " re.compile(r'^jcf.*?[0-9]{2}\\.cmap(\\.fits)?\\.gz$'", ' re.IGNORECASE)]', '']
["[re.compile(r'^jc.*?[0-9]{2}\\.cmap(\\.fits)?\\.gz$'", ' re.IGNORECASE)', " re.compile(r'^jc.*?[0-9]{2}\\.cmap(\\.fits)?\\.gz$'", ' re.IGNORECASE)]', '']
["[re.compile(r'^jc.*?[0-9]{2}\\.cmap(\\.fits)?\\.gz$'", ' re.IGNORECASE)', " re.compile(r'^jc.*?[0-9]{2}\\.cmap(\\.fits)?\\.gz$'", ' re.IGNORECASE)]', '']
["[re.compile(r'^jc.*?[0-9]{2}\\.cmap(\\.fits)?\\.gz$'", ' re.IGNORECASE)', " re.compile(r'^jc.*?[0-9]{2}\\.cmap(\\.fits)?\\.gz$'", ' re.IGNORECASE)]', '']
["[re.compile(r'^jcf.*?[0-9]{2}\\.cmap(\\.fits)?\\.gz$'", ' re.IGNORECASE)', " re.compile(r'^jcf.*?[0-9]{2}\\.cmap(\\.fits)?\\.gz$'", ' re.IGNORECASE)]', '']
["[re.compile(r'^jcf.*?[0-9]{2}\\.cmap(\\.fits)?\\.gz$'", 

In [17]:
print(row)

["[re.compile(r'^jcf.*?[0-9]{2}\\.cmap(\\.fits)?\\.gz$'", ' re.IGNORECASE)', " re.compile(r'^jcf.*?[0-9]{2}\\.cmap(\\.fits)?\\.gz$'", ' re.IGNORECASE)]']
